# Entity ***Tokens***
+ Layer bronze
+ Priority 1
---
### Imports

In [0]:
%run ../common/bronze_constants

In [0]:
%run ../common/medallion_functions

In [0]:
import requests
import time
from pyspark.sql.functions import current_timestamp

### Parameters

In [0]:
layer = dbutils.widgets.get("layer")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")
SUBGRAPH_URL = dbutils.widgets.get("SUBGRAPH_URL")

### Variables

In [0]:
tokens_list = []
query_variables = {
    "skip": 0
}

In [0]:
tokens_query = """ 
  query TokensQuery($skip: Int!){
    tokens(first: 1000, 
      skip: $skip,
      orderBy: volumeUSD, 
      orderDirection: desc
    ) {
      id
      symbol
      name
      decimals
      totalSupply
      volume
      volumeUSD
      feesUSD
      txCount
      poolCount
      totalValueLocked
      totalValueLockedUSD
      derivedETH
  }
}
"""

### Execution

In [0]:
while True:
    print(f"*INFO*: Count of skipped rows: {query_variables["skip"]}.")
    response = requests.post(SUBGRAPH_URL, json={"query": tokens_query, "variables": query_variables})
    tokens_data = response.json()["data"][entity_name]

    if "errors" in response.json():
            raise Exception(response.json()["errors"])

    tokens_list.extend(tokens_data)

    time.sleep(0.5)

    rows_loaded = len(tokens_data)
    print(f"*INFO*: Loaded rows: {rows_loaded}.")

    if rows_loaded < PAGE_SIZE:
        break
           
    query_variables["skip"] += PAGE_SIZE   

In [0]:
tokens_df = spark.createDataFrame(tokens_list)
tokens_df = tokens_df.withColumn("load_timestamp", current_timestamp())

### Save & exit

In [0]:
load_result = load_entity(tokens_df,
                        entity_name,
                        load_pattern,
                        layer
                        )

In [0]:
dbutils.notebook.exit(load_result)